In [1]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import json
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

/Users/anveshradharapu/Library/Caches/pypoetry/virtualenvs/mlengineerprep-fSSVlCLJ-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chroma_client = chromadb.Client()
model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight, good quality

In [3]:
documents = [
    "The sky is blue and beautiful.",
    "The sun is bright and hot.",
    "The ocean is vast and deep.",
    "Mountains are high and majestic.",
    "Rivers flow through valleys and cities."
]

ids = [f"doc{i+1}" for i in range(len(documents))]
embeddings = model.encode(documents)
print(embeddings.shape)
embeddings=embeddings.tolist()

(5, 384)


In [4]:
collection = chroma_client.get_or_create_collection(name="text_docs")

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids
)

In [5]:
query = "What is vast and bright?"
query_embedding = model.encode([query]).tolist()[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Top 3 matching documents:")
for doc in results['documents'][0]:
    print("-", doc)


Top 3 matching documents:
- The sun is bright and hot.
- The ocean is vast and deep.
- Mountains are high and majestic.


In [6]:
with open("../data/foodIngrediant.json", "r") as f:
    data = json.load(f)

# Step 2: Preprocess for embedding
def convert_to_text(doc):
    return (
        f"{doc['name']} is a {doc['cuisine']} dish. "
        f"It is made by {doc['method']} method and contains ingredients like {', '.join(doc['ingredients'])}. "
        f"It tastes {doc['taste']} and has {doc['calories']} calories. "
        f"{'Vegetarian' if not any(meat in doc['ingredients'] for meat in ['chicken', 'beef', 'pork', 'shrimp', 'salmon', 'mutton', 'anchovy']) else 'Non-vegetarian'}."
    )

documents = [convert_to_text(d) for d in data]


In [7]:
def clean_metadata(item):
    return {
        "name": item["name"],
        "cuisine": item["cuisine"],
        "calories": item["calories"],
        "ingredients": ", ".join(item["ingredients"]),  # <- flatten list
        "method": item["method"],
        "taste": item["taste"]
    }

In [8]:
metadatas = [clean_metadata(d) for d in data]

# Step 3: Initialize embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_9907/2606720017.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [9]:
import shutil, os

persist_dir = "../data/clean_food_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("✅ Removed existing clean_food_db")

os.makedirs(persist_dir, exist_ok=True)
print("📁 Created new clean_food_db directory at:", os.path.abspath(persist_dir))

vectorstore = Chroma(
    collection_name="foods",
    embedding_function=embedding_model,
    persist_directory=persist_dir
)

vectorstore = Chroma.from_texts(
    collection_name="foods",
    texts=documents,
    embedding=embedding_model,
    metadatas=metadatas
)

# Step 5: Create retriever
retriever = vectorstore.as_retriever()

✅ Removed existing clean_food_db
📁 Created new clean_food_db directory at: /Users/anveshradharapu/Documents/Project_Space/ML Engineer/MLEngineerPrep/IBM RAG and Agentic AI/data/clean_food_db


/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_9907/2520898134.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [10]:
!ollama list

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


NAME            ID              SIZE      MODIFIED   
gemma:latest    a72c7f4d0a15    5.0 GB    3 days ago    


In [11]:
from langchain.chains import RetrievalQA
from langchain_ollama import OllamaLLM  # or HuggingFacePipeline

llm = OllamaLLM(model="gemma")  # Make sure Ollama is running

qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
print(qa.invoke("Suggest a low-calorie vegetarian lunch with tomatoes."))


{'query': 'Suggest a low-calorie vegetarian lunch with tomatoes.', 'result': 'Based on the provided context, a low-calorie vegetarian lunch with tomatoes would be the **Tomato Quinoa Bowl**, with 280 calories.'}


In [12]:
docs = retriever.invoke("Suggest a low-calorie vegetarian lunch with tomatoes.")
for doc in docs:
    print("🔍 Retrieved:", doc.page_content[:200])

🔍 Retrieved: Tomato Quinoa Bowl is a Mediterranean dish. It is made by boiled method and contains ingredients like quinoa, tomatoes, spinach, lemon, olive oil. It tastes fresh and light and has 280 calories. Veget
🔍 Retrieved: Margherita Pizza is a Italian dish. It is made by baked method and contains ingredients like flour, tomato, mozzarella, basil. It tastes savory and has 500 calories. Vegetarian.
🔍 Retrieved: Greek Salad is a Greek dish. It is made by raw method and contains ingredients like feta cheese, cucumber, tomatoes, olives. It tastes savory and has 280 calories. Vegetarian.
🔍 Retrieved: Caesar Salad is a American dish. It is made by raw method and contains ingredients like lettuce, croutons, parmesan, anchovy. It tastes savory and has 330 calories. Non-vegetarian.


In [13]:
employees = [
    {
        "id": "emp1",
        "name": "Alice Johnson",
        "role": "Data Scientist",
        "skills": ["Python", "Machine Learning", "SQL"],
        "experience": "5 years",
        "location": "New York"
    },
    {
        "id": "emp2",
        "name": "Bob Smith",
        "role": "Software Engineer",
        "skills": ["Java", "Spring Boot", "AWS"],
        "experience": "4 years",
        "location": "San Francisco"
    },
    {
        "id": "emp3",
        "name": "Clara Lee",
        "role": "ML Engineer",
        "skills": ["Python", "Deep Learning", "TensorFlow"],
        "experience": "6 years",
        "location": "Boston"
    }
]


In [14]:
def employee_to_text(emp):
    return (
        f"{emp['name']} is a {emp['role']} with {emp['experience']} of experience, "
        f"skilled in {', '.join(emp['skills'])}, based in {emp['location']}."
    )

employee_docs = [employee_to_text(e) for e in employees]
ids = [e['id'] for e in employees]


In [15]:
embeddings = model.encode(employee_docs).tolist()
def clean_metadata(item):
    return {
        "id": item["id"],
        "name": item["name"],
        "role": item["role"],
        "experience": item["experience"],
        "location":item["location"],
        "skills": ", ".join(item["skills"]),  # <- flatten list
    }
metadata_e = [clean_metadata(d) for d in employees]

In [17]:
import shutil, os

persist_dir_e = "../data/clean_employee_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print("✅ Removed existing clean_employee_db")

os.makedirs(persist_dir, exist_ok=True)
print("📁 Created new clean_employee_db directory at:", os.path.abspath(persist_dir))

vectorstore_e = Chroma(
    collection_name="employees",
    embedding_function=embedding_model,
    persist_directory=persist_dir_e
)

vectorstore_e = Chroma.from_texts(
    collection_name="employees",
    texts=employee_docs,
    embedding=embedding_model,
    metadatas=metadata_e
)

# Step 5: Create retriever
retriever_e = vectorstore_e.as_retriever()

✅ Removed existing clean_employee_db
📁 Created new clean_employee_db directory at: /Users/anveshradharapu/Documents/Project_Space/ML Engineer/MLEngineerPrep/IBM RAG and Agentic AI/data/clean_food_db


In [18]:
qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever_e)
print(qa.invoke("Looking for a Python expert in machine learning"))

{'query': 'Looking for a Python expert in machine learning', 'result': 'Based on the provided context, **Alice Johnson** is the most suitable candidate for a Python expert in machine learning, as she has 5 years of experience in Machine Learning and is skilled in Python.'}


In [20]:
docs = retriever_e.invoke("Looking for a Python expert in machine learning")
for doc in docs:
    print("🔍 Retrieved:", doc.page_content[:200])

🔍 Retrieved: Alice Johnson is a Data Scientist with 5 years of experience, skilled in Python, Machine Learning, SQL, based in New York.
🔍 Retrieved: Clara Lee is a ML Engineer with 6 years of experience, skilled in Python, Deep Learning, TensorFlow, based in Boston.
🔍 Retrieved: Bob Smith is a Software Engineer with 4 years of experience, skilled in Java, Spring Boot, AWS, based in San Francisco.
